In [1]:
# configuración: imports y carga de archivos
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import KNeighborsClassifier
from sklearn.covariance import EllipticEnvelope
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve, roc_auc_score
import warnings; warnings.filterwarnings('ignore')

RUTA = (Path.home() / ".cache" / "kagglehub" / "datasets" / "libamariyam" / "ds2os-dataset" / "versions" / "1" / "DS2OS.csv")

df = pd.read_csv(RUTA, low_memory=False)
print(f'Dataset  : {RUTA.name}')
print(f'Shape    : {df.shape}')
anomaly_rate = (df['normality'] != 'normal').mean()
print(f'Anomalías: {anomaly_rate:.1%}')


Dataset  : DS2OS.csv
Shape    : (357952, 13)
Anomalías: 2.8%


In [2]:
# preprocesamiento (igual que CLASIFICADORES DS2OS)

# Guardar para secuencias Markovianas (antes de encoding/borrado de columnas)
df_seq = df[['sourceID', 'timestamp', 'operation']].copy()
df_seq['orig_idx'] = np.arange(len(df))

CLASS_MAP = {
    'normal'                        : 'Normal',
    'anomalous(DoSattack)'          : 'DoS',
    'anomalous(scan)'               : 'Scan',
    'anomalous(malitiousControl)'   : 'MaliciousControl',
    'anomalous(malitiousOperation)' : 'MaliciousOperation',
    'anomalous(spying)'             : 'Spying',
    'anomalous(dataProbing)'        : 'DataProbing',
    'anomalous(wrongSetUp)'         : 'WrongSetUp',
}
CLASS_TO_INT = {v: i for i, v in enumerate(CLASS_MAP.values())}

# Preprocesamiento igual que CLASIFICADORES DS2OS
df['accessedNodeType'] = df['accessedNodeType'].fillna('Malicious')
df['value'] = df['value'].replace({'False': 0, 'True': 1, 'Twenty': 20, 'none': 0})
df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0)

y_multi = df['normality'].map(CLASS_MAP).map(CLASS_TO_INT).values.astype(np.int8)
df = df.drop(columns=['timestamp', 'normality'])

CAT_COLS = [
    'sourceID', 'sourceAddress', 'sourceType', 'sourceLocation',
    'destinationServiceAddress', 'destinationServiceType',
    'destinationLocation', 'accessedNodeAddress', 'accessedNodeType', 'operation',
]
for col in CAT_COLS:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

FEAT_COLS = CAT_COLS + ['value']
X_full = df[FEAT_COLS].values.astype(np.float32)

# Etiqueta binaria para deteccion de anomalias (Normal=0, Ataque=1)
y_bin = (y_multi > 0).astype(int)

# NOTA: StandardScaler se aplica dentro del loop por semilla (post-split),
# igual que en CLASIFICADORES DS2OS

print(f'Features usadas ({len(FEAT_COLS)}): {FEAT_COLS}')
print(f'Normal (0): {(y_bin==0).sum():,}  |  Ataque (1): {(y_bin==1).sum():,}')
print(f'Anomalías : {y_bin.mean():.1%}')
print(f"\nOperaciones únicas: {df_seq['operation'].nunique()}")
print(df_seq['operation'].value_counts().to_string())


Features usadas (11): ['sourceID', 'sourceAddress', 'sourceType', 'sourceLocation', 'destinationServiceAddress', 'destinationServiceType', 'destinationLocation', 'accessedNodeAddress', 'accessedNodeType', 'operation', 'value']
Normal (0): 347,935  |  Ataque (1): 10,017
Anomalías : 2.8%

Operaciones únicas: 5
operation
read               248061
write              109648
lockSubtree           148
registerService        84
subscribe              11


In [3]:
# definición y Entrenamiento

def build_tm(sequences, order):
    counts = defaultdict(lambda: defaultdict(int))
    for seq in sequences:
        for t in range(order, len(seq)):
            counts[tuple(seq[t - order:t])][seq[t]] += 1
    return {ctx: {s: n / sum(v.values()) for s, n in v.items()}
            for ctx, v in counts.items()}

def compute_scores(sequences, tm, order, eps=1e-10):
    out = []
    for seq in sequences:
        for t, state in enumerate(seq):
            if t < order:
                out.append((0., 0., 0.)); continue
            ctx   = tuple(seq[t - order:t])
            trans = tm.get(ctx, {})
            p     = trans.get(state, eps)
            dist  = np.array(list(trans.values())) if trans else np.array([eps])
            dist /= dist.sum()
            out.append((
                -np.log(p + eps),
                -np.sum(dist * np.log(dist + eps)),
                np.log(p + eps)
            ))
    return np.array(out)

def markov_features(df_seq, idx_tr, idx_ev, order):
    def get_seqs(idx):
        d = df_seq.iloc[idx].sort_values(['sourceID', 'timestamp'])
        g = d.groupby('sourceID', sort=False)
        return [list(x['operation']) for _, x in g], [list(x['orig_idx']) for _, x in g]

    seqs_tr, _         = get_seqs(idx_tr)
    tm                 = build_tm(seqs_tr, order)
    seqs_ev, orig_idxs = get_seqs(idx_ev)
    scores             = compute_scores(seqs_ev, tm, order)
    flat_orig          = [i for grp in orig_idxs for i in grp]
    reorder            = np.argsort(flat_orig)
    s = scores[reorder]
    return pd.DataFrame({'surprise': s[:, 0], 'entropy': s[:, 1], 'loglik': s[:, 2]})

def youden_thr(y_true, sc):
    fpr, tpr, thr = roc_curve(y_true, sc)
    return thr[np.argmax(tpr - fpr)]

def get_models(seed):
    return {
        'RandomForest'   : RandomForestClassifier(n_estimators=50, random_state=seed, n_jobs=-1),
        'MLP'            : MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=50,
                                         early_stopping=True, n_iter_no_change=5,
                                         tol=1e-3, random_state=seed),
        'KNN'            : KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        'IsolationForest': IsolationForest(n_estimators=100, contamination='auto',
                                           random_state=seed, n_jobs=-1),
        'OCSVM'          : OneClassSVM(kernel='rbf', nu=0.1, gamma='scale'),
        'RobustCov'      : EllipticEnvelope(contamination=0.3, random_state=seed),
    }

UNSUP_SUB     = {'OCSVM', 'RobustCov'}
SEEDS         = list(range(10))
MARKOV_ORDERS = [0, 1, 2, 3, 4]

all_results = []
t0_total = time.time()

for seed in SEEDS:
    t0_seed = time.time()

    # Split estratificado 64 / 16 / 20 (test=20%, igual que CLASIFICADORES DS2OS)
    X_tv, X_te, y_tv, y_te, idx_tv, idx_te = train_test_split(
        X_full, y_bin, np.arange(len(y_bin)), test_size=0.20, random_state=seed, stratify=y_bin
    )
    X_tr, X_vl, y_tr, y_vl, idx_tr, idx_vl = train_test_split(
        X_tv, y_tv, idx_tv, test_size=0.20, random_state=seed, stratify=y_tv
    )

    # StandardScaler ajustado sobre train (post-split), igual que CLASIFICADORES DS2OS
    scaler  = StandardScaler()
    X_tr_s  = pd.DataFrame(scaler.fit_transform(X_tr), columns=FEAT_COLS)
    X_vl_s  = pd.DataFrame(scaler.transform(X_vl),     columns=FEAT_COLS)
    X_te_s  = pd.DataFrame(scaler.transform(X_te),     columns=FEAT_COLS)

    for k in MARKOV_ORDERS:
        t0_k = time.time()

        if k == 0:
            Xtr, Xvl, Xte = X_tr_s, X_vl_s, X_te_s
        else:
            mk_tr = markov_features(df_seq, idx_tr, idx_tr, k)
            mk_vl = markov_features(df_seq, idx_tr, idx_vl, k)
            mk_te = markov_features(df_seq, idx_tr, idx_te, k)
            Xtr = pd.concat([X_tr_s.reset_index(drop=True), mk_tr], axis=1)
            Xvl = pd.concat([X_vl_s.reset_index(drop=True), mk_vl], axis=1)
            Xte = pd.concat([X_te_s.reset_index(drop=True), mk_te], axis=1)

        Xtr_norm = Xtr[y_tr == 0]
        rng      = np.random.default_rng(seed)
        sub_idx  = rng.choice(len(Xtr_norm), size=min(5000, len(Xtr_norm)), replace=False)
        Xtr_sub  = Xtr_norm.iloc[sub_idx]

        for name, model in get_models(seed).items():
            t0_m = time.time()

            if name in ('RandomForest', 'MLP', 'KNN'):
                model.fit(Xtr, y_tr)
                sc_vl = model.predict_proba(Xvl)[:, 1]
                sc_te = model.predict_proba(Xte)[:, 1]
            else:
                fit_data = Xtr_sub if name in UNSUP_SUB else Xtr_norm
                model.fit(fit_data)
                score_fn = model.score_samples if hasattr(model, 'score_samples') else model.decision_function
                sc_vl = -score_fn(Xvl)
                sc_te = -score_fn(Xte)

            pred = (sc_te >= youden_thr(y_vl, sc_vl)).astype(int)
            all_results.append({
                'seed' : seed,
                'k'    : k,
                'model': name,
                'AUC'  : roc_auc_score(y_te, sc_te),
                'F1'   : f1_score(y_te, pred, zero_division=0),
                'Prec' : precision_score(y_te, pred, zero_division=0),
                'Rec'  : recall_score(y_te, pred, zero_division=0),
            })
            print(f'  seed={seed} k={k} {name:<18} {time.time()-t0_m:5.1f}s', flush=True)

        print(f'  --- k={k} completado en {time.time()-t0_k:.1f}s ---', flush=True)

    elapsed   = time.time() - t0_seed
    remaining = elapsed * (len(SEEDS) - seed - 1)
    print(f'Semilla {seed} completada en {elapsed:.0f}s | ETA restante: {remaining/60:.1f} min', flush=True)
    print()


  seed=0 k=0 RandomForest         2.5s
  seed=0 k=0 MLP                 16.9s
  seed=0 k=0 KNN                 26.2s
  seed=0 k=0 IsolationForest      0.9s
  seed=0 k=0 OCSVM                3.9s
  seed=0 k=0 RobustCov            0.4s
  --- k=0 completado en 50.9s ---
  seed=0 k=1 RandomForest         1.8s
  seed=0 k=1 MLP                  7.6s
  seed=0 k=1 KNN                 21.6s
  seed=0 k=1 IsolationForest      0.9s
  seed=0 k=1 OCSVM                3.9s
  seed=0 k=1 RobustCov            0.4s
  --- k=1 completado en 40.3s ---
  seed=0 k=2 RandomForest         2.1s
  seed=0 k=2 MLP                 13.6s
  seed=0 k=2 KNN                 20.5s
  seed=0 k=2 IsolationForest      0.9s
  seed=0 k=2 OCSVM                3.9s
  seed=0 k=2 RobustCov            0.4s
  --- k=2 completado en 45.6s ---
  seed=0 k=3 RandomForest         1.8s
  seed=0 k=3 MLP                  9.5s
  seed=0 k=3 KNN                 26.3s
  seed=0 k=3 IsolationForest      0.9s
  seed=0 k=3 OCSVM                3.8s
 

In [4]:
# resultados: métricas sobre el set de test

df_res = pd.DataFrame(all_results)
names  = list(df_res['model'].unique())

W     = 102
TITLE = 'TABLA DE RESULTADOS — AUC y F1-Score por modelo y orden Markoviano'
col_modelo   = 'Modelo'
col_orden    = 'Orden'
col_auc      = 'AUC media'
col_std      = '±std'
col_f1       = 'F1 media'
col_prec     = 'Precisión'
col_rec      = 'Recall'
HDR = (f'{col_modelo:<18} {col_orden:>5}   {col_auc:>10}  {col_std:>6}   '
       f'{col_f1:>9}  {col_std:>6}   {col_prec:>9}  {col_rec:>7}   Tend.')
SEP = '-' * W

print('=' * W)
print(TITLE.center(W))
print('=' * W)
print(HDR)
print(SEP)

for i, name in enumerate(names):
    aucs = [df_res[(df_res['model'] == name) & (df_res['k'] == k)]['AUC'].mean()
            for k in MARKOV_ORDERS]
    d = aucs[-1] - aucs[0]

    if   max(aucs) - min(aucs) < 0.001:                              tend = 'estable'
    elif all(aucs[j] >= aucs[j-1] for j in range(1, len(aucs))):     tend = f'mejora  d=+{d:.4f}'
    elif all(aucs[j] <= aucs[j-1] for j in range(1, len(aucs))):     tend = f'degrada d={d:.4f}'
    else:                                                             tend = f'mixta   d={d:+.4f}'

    for k in MARKOV_ORDERS:
        sub  = df_res[(df_res['model'] == name) & (df_res['k'] == k)]
        r    = {m: sub[m].mean() for m in ('AUC', 'F1', 'Prec', 'Rec')}
        rs   = {m: sub[m].std()  for m in ('AUC', 'F1')}

        if   k == 0:                            arrow = '-'
        elif aucs[k] > aucs[k-1] + 0.0005:     arrow = 'sube'
        elif aucs[k] < aucs[k-1] - 0.0005:     arrow = 'baja'
        else:                                   arrow = '='

        tend_col = tend if k == 0 else ''
        kstr = f'k={k}'
        print(f"{name:<18} {kstr:>5}   {r['AUC']:>10.4f}  {rs['AUC']:>6.4f}   "
              f"{r['F1']:>9.4f}  {rs['F1']:>6.4f}   "
              f"{r['Prec']:>9.4f}  {r['Rec']:>7.4f}   {arrow:<5}  {tend_col}")

    if i < len(names) - 1:
        print()
        print(SEP)

print('=' * W)

print()
print('=' * 72)
titulo_resumen = 'RESUMEN DE TENDENCIAS Y VEREDICTO'
print(f'{titulo_resumen:^72}')
print('=' * 72)
for name in names:
    aucs   = [df_res[(df_res['model'] == name) & (df_res['k'] == k)]['AUC'].mean()
              for k in MARKOV_ORDERS]
    f1s    = [df_res[(df_res['model'] == name) & (df_res['k'] == k)]['F1'].mean()
              for k in MARKOV_ORDERS]
    best_k = int(np.argmax(aucs))
    d_auc  = aucs[-1] - aucs[0]
    print(f'\n  {name}')
    print('    AUC por orden: ' + '  '.join(f'k={k}:{aucs[k]:.4f}' for k in MARKOV_ORDERS))
    print('    F1  por orden: ' + '  '.join(f'k={k}:{f1s[k]:.4f}'  for k in MARKOV_ORDERS))
    print(f'    Mejor k       : k={best_k}  AUC={aucs[best_k]:.4f}  F1={f1s[best_k]:.4f}')
    if   d_auc > 0.002:   verdict = 'Markov AYUDA'
    elif d_auc < -0.002:  verdict = 'Markov PERJUDICA'
    else:                 verdict = 'Markov SIN EFECTO CLARO'
    print(f'    Veredicto     : {verdict}  (delta AUC k0->k4 = {d_auc:+.4f})')


                  TABLA DE RESULTADOS — AUC y F1-Score por modelo y orden Markoviano                  
Modelo             Orden    AUC media    ±std    F1 media    ±std   Precisión   Recall   Tend.
------------------------------------------------------------------------------------------------------
RandomForest         k=0       0.9988  0.0001      0.8394  0.0254      0.7631   0.9603   -      estable
RandomForest         k=1       0.9988  0.0001      0.8306  0.0061      0.7172   0.9876   =      
RandomForest         k=2       0.9987  0.0001      0.8278  0.0056      0.7064   0.9997   =      
RandomForest         k=3       0.9987  0.0002      0.8263  0.0075      0.7042   0.9998   =      
RandomForest         k=4       0.9985  0.0004      0.8230  0.0077      0.6996   0.9995   =      

------------------------------------------------------------------------------------------------------
MLP                  k=0       0.9987  0.0001      0.8278  0.0060      0.7062   1.0000   -      estable